# PROYECTO DE APRENDIZAJE AUTOMÁTICO: SIMULADOR PREDICTIVO DE PIT STOPS (F1 2025)
* **Materia:** Aprendizaje Automático / Ciencia de Datos (2026)
* **Estudiante:** Martín Ozuna
* **Enfoque del Modelo:** Regresión Supervisada Basada en Árboles para emular búsquedas reales de ventanas de paradas.

In [15]:
import os
import pandas as pd
import numpy as np

# Configuración de paths globales del repositorio estructurado
PATH_RAW = os.path.join('..', 'data', 'raw')
PATH_PROCESSED = os.path.join('..', 'data', 'processed')

print("✓ Pipeline Inicializado. Entorno Cookiecutter vinculado correctamente.")

✓ Pipeline Inicializado. Entorno Cookiecutter vinculado correctamente.


## 1. Módulo de Ingesta de Datos (Ecosistema Multifuente)
En esta sección se centraliza la entrada de datos crudos. Está diseñado para recibir tanto los archivos relacionales estáticos como las futuras llamadas a APIs meteorológicas y de telemetría sin alterar las fases de modelado.

In [16]:
# ==============================================================================
# CELDA 4: INGESTA LOCAL Y FILTRADO EXCLUSIVO - TEMPORADA DE REFERENCIA 2024
# ==============================================================================

try:
    # Carga de los 5 archivos seleccionados de tu carpeta raw
    races = pd.read_csv(os.path.join(PATH_RAW, 'races.csv'))
    pit_stops = pd.read_csv(os.path.join(PATH_RAW, 'pit_stops.csv'))
    lap_times = pd.read_csv(os.path.join(PATH_RAW, 'lap_times.csv'))
    drivers = pd.read_csv(os.path.join(PATH_RAW, 'drivers.csv'))
    circuits = pd.read_csv(os.path.join(PATH_RAW, 'circuits.csv'))
    
    print("✓ Los 5 archivos esenciales se cargaron correctamente desde data/raw/")
    
    # AISLAMIENTO HISTÓRICO: Cambiamos a 2024 para tener datos reales e instancias pobladas
    races_2024 = races[races['year'] == 2024]
    races_ids_2024 = races_2024['raceId'].unique()
    
    # Filtrado en cascada para el espacio muestral
    pit_stops_2024 = pit_stops[pit_stops['raceId'].isin(races_ids_2024)]
    lap_times_2024 = lap_times[lap_times['raceId'].isin(races_ids_2024)]
    
    print(f"\n--- Resumen de Ingesta Acotada (Temporada de Referencia 2024) ---")
    print(f"Grandes Premios identificados en 2024: {races_2024.shape[0]}")
    print(f"Registros de Pit Stops capturados: {pit_stops_2024.shape[0]}")
    print(f"Historial de tiempos de vuelta cargados: {lap_times_2024.shape[0]}")

except FileNotFoundError as e:
    print(f"[ERROR] No se encontró algún archivo en data/raw/. Verificá la extracción. Detalle: {e}")

✓ Los 5 archivos esenciales se cargaron correctamente desde data/raw/

--- Resumen de Ingesta Acotada (Temporada de Referencia 2024) ---
Grandes Premios identificados en 2024: 24
Registros de Pit Stops capturados: 825
Historial de tiempos de vuelta cargados: 26574


## 2. Consolidación de la Matriz de Datos Semilla
Aquí cruzamos las tablas relacionales utilizando los IDs de carrera, piloto y circuito. El resultado de esta celda es nuestro dataset maestro crudo unificado sobre el cual realizaremos la descripción técnica solicitada para la Entrega 2.

In [17]:
# ==============================================================================
# CELDA 6: MERGE ESTRUCTURAL DE PILOTOS, CIRCUITOS Y TIEMPOS (BASE MAESTRA)
# ==============================================================================

# Unimos los tiempos de vuelta con los metadatos correspondientes del 2024
df_f1_2024_raw = pd.merge(lap_times_2024, races_2024[['raceId', 'circuitId', 'name', 'round']], on='raceId', how='inner')
df_f1_2024_raw = pd.merge(df_f1_2024_raw, drivers[['driverId', 'driverRef']], on='driverId', how='inner')
df_f1_2024_raw = pd.merge(df_f1_2024_raw, circuits[['circuitId', 'circuitRef']], on='circuitId', how='inner')

# Cambiamos los nombres para mayor claridad del diccionario
df_f1_2024_raw.rename(columns={'name': 'grand_prix_name', 'driverRef': 'driver_name', 'circuitRef': 'circuit_name'}, inplace=True)

print(f"✓ Matriz semilla 'df_f1_2024_raw' construida con éxito.")
print(f"Dimensiones reales del dataset: {df_f1_2024_raw.shape[0]} filas (instancias) x {df_f1_2024_raw.shape[1]} columnas.")


✓ Matriz semilla 'df_f1_2024_raw' construida con éxito.
Dimensiones reales del dataset: 26574 filas (instancias) x 11 columnas.


## 3. Descripción del Dataset y Diccionario de Datos (Requisito Entrega 2)
Espacio dedicado a la auditoría exigida por la cátedra para controlar los tipos de datos, inspeccionar nulos e identificar formalmente nuestra variable target continua.

In [18]:
# ==============================================================================
# SUBTAREA: METADATOS PARA EL INFORME DE LA CÁTEDRA
# ==============================================================================
df_f1_2024_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26574 entries, 0 to 26573
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   raceId           26574 non-null  int64 
 1   driverId         26574 non-null  int64 
 2   lap              26574 non-null  int64 
 3   position         26574 non-null  int64 
 4   time             26574 non-null  object
 5   milliseconds     26574 non-null  int64 
 6   circuitId        26574 non-null  int64 
 7   grand_prix_name  26574 non-null  object
 8   round            26574 non-null  int64 
 9   driver_name      26574 non-null  object
 10  circuit_name     26574 non-null  object
dtypes: int64(7), object(4)
memory usage: 2.2+ MB


Celda 9: RESGUARDO METODOLÓGICO PARA FUTURAS ENTREGAS (Markdown)
## 4. Módulos Reservados para Desarrollo Futuro (Próximas Entregas)
Las siguientes secciones quedan planteadas de manera estructural. Al estar separadas, las correcciones del profesor sobre limpieza o ingeniería no afectarán la ingesta realizada el día de hoy.

### 4.1. Módulo de Preprocesamiento y Curación de Datos
*Remoción de anomalías estocásticas (Safety Cars, DNF por choques, roturas mecánicas).*

### 4.2. Módulo de Feature Engineering e Inserción de Nuevas Fuentes
*Acoplamiento de datos climáticos externos y cálculo de la edad física del neumático por FastF1.*

### 4.3. Módulo de Modelado de Regresión Supervisada
*Entrenamiento comparativo: Regresión Lineal Múltiple, DecisionTreeRegressor y RandomForestRegressor.*